In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

### Confirm data time series overlap with 12442.  If data time series is the same then delete, if not then concatenate with 12442 into the Additional PRPA station (ENV Native ID 547) and delete

### Confirm data time series overlap with 12568.  If data time series is the same then delete 12568, if not then concatenate with 12568 into the Additional PRPA station (ENV Native ID 547) and delete.

### Additional (see stations 12568 and 12442)

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

wanted_issues = [
"Confirm data time series overlap with 12442.  If data time series is the same then delete, if not then concatenate with 12442 into the Additional PRPA station (ENV Native ID 547) and delete"]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_concat =  df_concat[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_concat["Network ID"] = pd.to_numeric(df_concat["Network ID"], errors="coerce").astype("Int64")
df_concat["Station ID"] = pd.to_numeric(df_concat["Station ID"], errors="coerce").astype("Int64")

df_concat['check_station_id'] = ['12442']
df_concat




,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442


In [6]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]


stats = df_concat.apply(
    lambda r: count_station_stats(
        int(r["Station ID"]),
        int(r["check_station_id"]),
        engine
    ),
    axis=1
)

df_concat[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442,5660,175774,1148,1148


In [7]:
query = """
SELECT station_id
FROM meta_station
WHERE native_id = %s
"""

def get_new_native_id_from_s(native_id,engine):
    # handle <NA> safely
    if pd.isna(native_id):
        return None

    df = pd.read_sql(
        query,
        engine,
        params=(str(native_id),)
    )

    if df.empty:
        return None

    return df.iloc[0]["station_id"]  # TEXT → string

new_native_id = 547

df_concat["new_station_id_from_s"] = df_concat.apply(
    lambda row: get_new_native_id_from_s(
        new_native_id,
        engine
    ),
    axis=1
)
df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_old,n_new,n_overlap,n_overlap_same_datum,new_station_id_from_s
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442,5660,175774,1148,1148,2009


Seems the new native_id 547, exited in the current PCDS but under network 12 (FLNRO-WMB). So not sure how concatnate into the Additional PRPA station (ENV Native ID 547) (PRPA seems like the new station want to included).

Need to insert the new network, then new station, then concatnate data into the new station, then delete both stations

In [15]:

wanted_issues = [
    'Additional (see stations 12568 and 12442']

df_add = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_add = df_add[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

pattern = r'->\s*([0-9\.\-]+)\s*W,\s*([0-9\.\-]+)\s*N,\s*Elev\.\s*([0-9\.\-]+)\s*m'

df_add[['lon', 'lat', 'elev']] = df_add['Unique Location (String)'].str.extract(pattern).astype(float)

df_add["Native ID"] = df_add["Native ID"].str.replace(r'.*->\s*', '', regex=True)
df_add = df_add.drop(columns=['Unique Location (String)'])
df_add["Network Name"] = df_add["Network Name"].str.replace(r'.*->\s*', '', regex=True)


df_add['Station_name'] = df_add['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_add = df_add.drop(columns=['Unique Names'])
df_add.iloc[0]

from sqlalchemy import text

SQL_GET_NETWORK_ID = text("""
SELECT network_id
FROM meta_network
WHERE LOWER(network_name) = LOWER(:network_name)
""")

def get_network_id(network_name, engine):
    """Return the network_id for a given network_name."""
    with engine.connect() as conn:
        result = conn.execute(SQL_GET_NETWORK_ID, {"network_name": network_name}).scalar()
    return result

# Make sure 'Network Name' column exists
df_add['Network ID'] = df_add['Network Name'].apply(lambda name: get_network_id(name, engine))
df_add


,ISSUE,Station ID,Network ID,Network Name,Native ID,lon,lat,elev,Station_name
0,Additional (see stations 12568 and 12442,NaN,76,PRPA,547,130.352188,54.292615,15.0,Prince Rupert Fairview


In [16]:
from sqlalchemy import func

stations_created = []

# for _, row in df.iloc[0:2].iterrows():
for _, row in df_add.iterrows():
    
    name = row['Station_name']
    nid  = row['Native ID']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['elev'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Prince Rupert Fairview', 14956)]


In [24]:
stations_created

[('Prince Rupert Fairview', 14956)]

In [20]:
from sqlalchemy import text

# Delete duplicates between two history_id's
SQL_DELETE_DUPLICATES = text("""
DELETE FROM obs_raw o
USING obs_raw o2
WHERE o.vars_id = o2.vars_id
  AND o.obs_time = o2.obs_time
  AND o.history_id = :old_hist_id
  AND o2.history_id = :new_hist_id
""")

# Bulk move obs_raw to new history_id
SQL_BULK_MOVE = text("""
UPDATE obs_raw
SET history_id = :new_hist_id
WHERE history_id = :old_hist_id
""")

# Get the latest history_id for a given station_id
SQL_GET_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

def move_station_data_fast_by_id(old_station_id, new_station_id, engine):
    with engine.begin() as conn:

        old_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {"station_id": old_station_id}
        ).scalar()

        new_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {"station_id": new_station_id}
        ).scalar()

        if old_hist_id is None or new_hist_id is None:
            print(f"❌ Missing history: {old_station_id} → {new_station_id}")
            return 0

        # 1️⃣ delete duplicates
        dup_res = conn.execute(
            SQL_DELETE_DUPLICATES,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        # 2️⃣ bulk move
        move_res = conn.execute(
            SQL_BULK_MOVE,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        print(
            f"Station {old_station_id} → {new_station_id}: "
            f"🧹 removed duplicates={dup_res.rowcount}, "
            f"🚚 moved={move_res.rowcount}"
        )

        return move_res.rowcount

# Example: move one station
old_station_id1 = 12442
old_station_id2 = 12568
new_station_id = stations_created[0][1]

move_station_data_fast_by_id(old_station_id1, new_station_id, engine)
move_station_data_fast_by_id(old_station_id2, new_station_id, engine)

Station 12442 → 14956: 🧹 removed duplicates=0, 🚚 moved=102960
Station 12568 → 14956: 🧹 removed duplicates=1148, 🚚 moved=4512


4512

### Check the new station's record

In [ ]:
query = """
SELECT COUNT(*) AS n_obs
FROM meta_history m
JOIN obs_raw o ON m.history_id = o.history_id
WHERE m.station_id = %s
"""

df = pd.read_sql(query, engine, params=(new_station_id,))
print(df.iloc[0]["n_obs"])

180286


### update variable id

In [26]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_INFO = text("""
SELECT DISTINCT
       -- o.vars_id,
       v.*,
       -- m.station_id,
       s.native_id
       -- m.station_name,
       -- s.network_id AS station_network_id
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN obs_raw AS o ON m.history_id = o.history_id
JOIN meta_vars AS v ON o.vars_id = v.vars_id
WHERE s.station_id = :station_id
""")

old_station_vars = pd.read_sql(
    SQL_VARS_INFO,
    engine,
    params={"station_id": new_station_id
}
)

old_station_vars

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id
0,472,9,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean,2025-02-11 16:03:39.747374,crmp,547
1,477,9,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,2025-02-11 16:03:39.747374,crmp,547
2,693,9,celsius,None,air_temperature,time: mean,None,avg_air_temp_pst1hr,Temperature (Mean),T,2025-02-11 16:03:39.747374,crmp,547
3,474,9,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point,2025-02-11 16:03:39.747374,crmp,547
4,476,9,degree,None,wind_from_direction,time: point,None,WDIR_VECT,Wind Direction (Point),wind_from_direction_point,2025-02-11 16:03:39.747374,crmp,547
5,699,9,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,2025-02-11 16:03:39.747374,crmp,547


In [36]:
from sqlalchemy import text
import pandas as pd

SQL_GET_NETWORK_NAME = text("""
SELECT n.network_name, n.network_id
FROM meta_station s
JOIN meta_network n ON s.network_id = n.network_id
WHERE s.station_id = :station_id
LIMIT 1
""")

SQL_VARS_new = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name
FROM meta_vars v
JOIN meta_network n ON v.network_id = n.network_id
WHERE n.network_name = :network_name
""")

# Use a connection context
with engine.connect() as conn:
    row = conn.execute(SQL_GET_NETWORK_NAME, {"station_id": new_station_id}).mappings().first()
    if row is None:
        raise ValueError(f"Station {new_station_id} not found")

    network_name = row["network_name"]
    network_id = row["network_id"]
    print("Network name:", network_name, "Network ID:", network_id)

    new_net_vars = pd.read_sql(
        SQL_VARS_new,
        conn,
        params={"network_name": network_name}
    )

new_net_vars.head()

Network name: PRPA Network ID: 76


,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name
0,1270,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean
1,1271,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point
2,1272,celsius,None,air_temperature,time: mean,None,avg_air_temp_pst1hr,Temperature (Mean),T
3,1273,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point
4,1274,degree,None,wind_from_direction,time: point,None,WDIR_VECT,Wind Direction (Point),wind_from_direction_point


In [37]:
VAR_MATCH_COLS = [
    "unit",
    "precision",
    "standard_name",
    "cell_method",
    "long_description",
    "net_var_name",
    "display_name",
    "short_name",
]

# Merge again, but keep vars_id from network 14
df_compare = old_station_vars.merge(
    new_net_vars[VAR_MATCH_COLS + ["vars_id"]],  # only keep vars_id from new_net
    on=VAR_MATCH_COLS,
    how="left",
    suffixes=("", "_new_net"),
    indicator=True
)

# Add a boolean column to mark existence
df_compare["exists_in_new_net"] = df_compare["_merge"] == "both"

# Rename vars_id from network 14 for clarity
df_compare["vars_id_new_net"] = df_compare["vars_id_new_net"]



,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id,vars_id_new_net,_merge,exists_in_new_net
0,472,9,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean,2025-02-11 16:03:39.747374,crmp,547,1270,both,True
1,477,9,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,2025-02-11 16:03:39.747374,crmp,547,1271,both,True
2,693,9,celsius,None,air_temperature,time: mean,None,avg_air_temp_pst1hr,Temperature (Mean),T,2025-02-11 16:03:39.747374,crmp,547,1272,both,True
3,474,9,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point,2025-02-11 16:03:39.747374,crmp,547,1273,both,True
4,476,9,degree,None,wind_from_direction,time: point,None,WDIR_VECT,Wind Direction (Point),wind_from_direction_point,2025-02-11 16:03:39.747374,crmp,547,1274,both,True
5,699,9,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,2025-02-11 16:03:39.747374,crmp,547,1275,both,True


In [ ]:
from pycds import Variable

vars_created = []
vars_map = {}   # old_vars_id → new_vars_id

for _, row in df_compare.iterrows():

    v = Variable(
        network_id=network_id,
        unit=row["unit"],
        precision=row["precision"],
        standard_name=row["standard_name"],
        cell_method=row["cell_method"],
        description=row["long_description"],
        name=row["net_var_name"],
        display_name=row["display_name"],
        short_name=row["short_name"],
    )

    session.add(v)
    session.flush()   # assigns vars_id

    vars_created.append(v.id)
    vars_map[row["vars_id"]] = v.id   # ← critical for later obs reassignment

session.commit()

print(f"Created {len(vars_created)} variables for network {network_id}")

IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "network_variable_name_unique"
DETAIL:  Key (network_id, net_var_name)=(76, TEMP_MEAN) already exists.

[SQL: INSERT INTO crmp.meta_vars (net_var_name, unit, standard_name, cell_method, precision, long_description, display_name, short_name, network_id) VALUES (%(net_var_name)s, %(unit)s, %(standard_name)s, %(cell_method)s, %(precision)s, %(long_description)s, %(display_name)s, %(short_name)s, %(network_id)s) RETURNING crmp.meta_vars.vars_id, crmp.meta_vars.mod_time, crmp.meta_vars.mod_user]
[parameters: {'net_var_name': 'TEMP_MEAN', 'unit': 'celsius', 'standard_name': 'air_temperature', 'cell_method': 'time: mean', 'precision': None, 'long_description': 'Present temperature', 'display_name': 'Temperature (Mean)', 'short_name': 'air_temperature_mean', 'network_id': 76}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

### Update the vars id to the new one

In [ ]:
import json
from sqlalchemy import text

# pairs_json = json.dumps(
#     [{"old_vars_id": old, "new_vars_id": new}
#      for old, new in vars_map.items()]
# )

df_pairs = df_compare.dropna(subset=["vars_id_new_net"])

pairs_json = json.dumps([
    {"old_vars_id": old, "new_vars_id": int(new)}
    for old, new in zip(df_pairs["vars_id"], df_pairs["vars_id_new_net"])
])

SQL_UPDATE_VARS_BULK = text("""
UPDATE obs_raw o
SET vars_id = m.new_vars_id
FROM meta_history h
JOIN meta_station s
  ON h.station_id = s.station_id
CROSS JOIN json_to_recordset(:pairs_json)
     AS m(old_vars_id INT, new_vars_id INT)
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND o.vars_id = m.old_vars_id
""")

new_station_id = new_station_id

with engine.begin() as conn:
    res = conn.execute(
        SQL_UPDATE_VARS_BULK,
        {
            "pairs_json": pairs_json,
            "station_id": new_station_id,
        }
    )

print(
    f"✅ vars_id remap completed | "
    f"station={new_station_id} | "
    f"mappings={len(vars_map)} | "
    f"rows_updated={res.rowcount}"
)

✅ vars_id remap completed | station=14956 | mappings=0 | rows_updated=180286


### delete old station data

In [22]:
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12442

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12442: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


In [23]:
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12568

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12568: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


### Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete.


In [8]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete."]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_concat =  df_concat[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_concat["Network ID"] = pd.to_numeric(df_concat["Network ID"], errors="coerce").astype("Int64")
df_concat["Station ID"] = pd.to_numeric(df_concat["Station ID"], errors="coerce").astype("Int64")

df_concat['check_station_id'] = ['12454']
df_concat



,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id
0,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,Crofton Elementary -> 564,NA -> Crofton Elementary,12454


In [9]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    WHERE o.history_id = h_old.history_id
      AND h_old.station_id = :old_station_id
      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          WHERE h_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_station_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:

        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": int(new_station_id)}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_station_id": int(old_station_id),
                "new_station_id": int(new_station_id),
                "new_hist_id": int(new_hist_id),
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

results = []

for _, row in df_concat.iterrows():
    n = move_station_obs_fast(
        row["Station ID"],
        row["check_station_id"],
        engine
    )
    results.append(n)

df_concat["n_moved"] = results

Moved 4490 rows from station 12541 → 12454


In [10]:
df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_moved
0,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,Crofton Elementary -> 564,NA -> Crofton Elementary,12454,4490


In [13]:
select_sql = sa.text("""
SELECT DISTINCT
    n.network_id
FROM meta_station s
JOIN meta_network n
  ON n.network_id = s.network_id
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.station_id = :station_id
""")

with engine.begin() as conn:
    for idx, row in df_concat.iterrows():
        # get the value from the row, not the entire Series
        station_id = row["check_station_id"]  # <-- this is a scalar

        result = conn.execute(
            select_sql,
            {"station_id": station_id}
        ).fetchall()

        if not result:
            print(f"❌ station_id {station_id} not found")
            continue

        df_concat.at[idx, "new_network_id"] = ",".join(
            map(str, {r.network_id for r in result})
        )

df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_moved,new_network_id
0,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,Crofton Elementary -> 564,NA -> Crofton Elementary,12454,4490,9


### The old station can be deleted

In [14]:
from tqdm import tqdm
import sqlalchemy as sa

station_ids = (
    df_concat["Station ID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
# station_ids[0]


# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]

Station 12541: flags=0, obs_raw=1148, obs_derived=0, meta_history=1, meta_station=1


In [ ]:
### How about the native_id change and name_change?